# Deep Learning for Email Classification with LSTM and Word2vec

## Task 1: Import Libraries

In [1]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

2026-05-04 06:32:56.575976: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Task 2: Load the Dataset

In [2]:
# Load the dataset and specify the correct encoding when reading the CSV file
df = pd.read_csv('/usercode/Dataset.csv', encoding='latin1')

# Print the DataFrame head
print(df.head())

  Label                                              Email
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


## Task 3: Extract Email Texts and Labels

In [3]:
# Extract 'texts' and 'labels'
texts = df['Email'].tolist()
labels = df['Label'].map({'ham': 0, 'spam': 1}).tolist()

# Print the total number of spam and ham emails
print("Total no. of spam emails:", sum(labels))
print("Total no. of ham emails:", len(labels) - sum(labels))

Total no. of spam emails: 747
Total no. of ham emails: 4825


## Task 4: Split the Dataset

In [4]:
# Split the dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(texts, labels, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

## Task 5: Tokenize and Pad Sequences

In [5]:
# Tokenize and pad sequences for training, validation, and testing
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train + X_val + X_test)

sequences_train = tokenizer.texts_to_sequences(X_train)
sequences_val = tokenizer.texts_to_sequences(X_val)
sequences_test = tokenizer.texts_to_sequences(X_test)

max_sequence_length = max([len(seq) for seq in sequences_train + sequences_val + sequences_test])
vocab_size = len(tokenizer.word_index) + 1

data_train = pad_sequences(sequences_train, maxlen=max_sequence_length)
data_val = pad_sequences(sequences_val, maxlen=max_sequence_length)
data_test = pad_sequences(sequences_test, maxlen=max_sequence_length)

## Task 6: Train a Word2Vec Model

In [6]:
sentences = [text.split() for text in X_train + X_val + X_test]
word2vec_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=1, workers=4)

## Task 7: Prepare the Embedding Matrix

In [7]:
embedding_matrix = np.zeros((vocab_size, word2vec_model.vector_size))
for word, i in tokenizer.word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]

## Task 8: Build an LSTM Model

In [8]:
# Build an LSTM model with Word2Vec embeddings
model = Sequential()
model.add(Embedding(vocab_size, 
word2vec_model.vector_size, weights=[embedding_matrix], 
input_length=max_sequence_length, trainable=False))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 189, 100)          892100    
                                                                 
 lstm (LSTM)                 (None, 100)               80400     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 972601 (3.71 MB)
Trainable params: 80501 (314.46 KB)
Non-trainable params: 892100 (3.40 MB)
_________________________________________________________________


## Task 9: Compile the Model

In [9]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

## Task 10: Train the Model

In [10]:
model.fit(data_train, np.array(y_train), epochs=10, batch_size=32, validation_data=(data_val, np.array(y_val)))

Epoch 1/10
122/122 [==============================] - 40s 303ms/step - loss: 0.3099 - accuracy: 0.8849 - val_loss: 0.2469 - val_accuracy: 0.9019
Epoch 2/10
122/122 [==============================] - 36s 297ms/step - loss: 0.2550 - accuracy: 0.8982 - val_loss: 0.2169 - val_accuracy: 0.9163
Epoch 3/10
122/122 [==============================] - 36s 299ms/step - loss: 0.2370 - accuracy: 0.9023 - val_loss: 0.2021 - val_accuracy: 0.9187
Epoch 4/10
122/122 [==============================] - 36s 296ms/step - loss: 0.2365 - accuracy: 0.9044 - val_loss: 0.2111 - val_accuracy: 0.9187
Epoch 5/10
122/122 [==============================] - 36s 291ms/step - loss: 0.2278 - accuracy: 0.9072 - val_loss: 0.1937 - val_accuracy: 0.9127
Epoch 6/10
122/122 [==============================] - 36s 292ms/step - loss: 0.2250 - accuracy: 0.9103 - val_loss: 0.2259 - val_accuracy: 0.9151
Epoch 7/10
122/122 [==============================] - 37s 300ms/step - loss: 0.2213 - accuracy: 0.9087 - val_loss: 0.1916 - val_ac

## Task 11: Evaluate the Model

In [11]:
evaluation_results = model.evaluate(data_test, np.array(y_test))
print("Test Loss:", evaluation_results[0])
print("Test Accuracy:", evaluation_results[1])

27/27 [==============================] - 1s 45ms/step - loss: 0.2441 - accuracy: 0.8959
Test Loss: 0.24407197535037994
Test Accuracy: 0.8959330320358276


## Task 12: Generate Predictions

In [12]:
# Generate predictions on the test set
predictions = model.predict(data_test)
predictions = (predictions > 0.5).astype(int)

27/27 [==============================] - 1s 44ms/step


## Task 13: Print the Classification Report

In [13]:
print("Classification Report:")
print(classification_report(np.array(y_test), predictions))

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.97      0.94       724
           1       0.68      0.42      0.52       112

    accuracy                           0.90       836
   macro avg       0.80      0.69      0.73       836
weighted avg       0.88      0.90      0.89       836

